<a href="https://colab.research.google.com/github/Raffy0-1/DHC-Phase-2-ML-Task_1/blob/main/News_Topic_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Requirements

In [ ]:
 pip install "transformers>=4.35.0" "datasets>=2.14.0" "torch>=2.0.0" "scikit-learn>=1.3.0" "gradio>=4.0.0" "numpy>=1.24.0" "accelerate>=0.24.0"

#Phase 1: Exploring The AG News Data

Imports

In [ ]:
from datasets import load_dataset
from collections import Counter

Loading Dataset

In [ ]:
print("Loading AG News dataset...")
data = load_dataset("ag_news")

Loading AG News dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:
print("\n─── Dataset Structure ───")
print(data)


─── Dataset Structure ───
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


Peeking at a few Examples

In [ ]:
print("\n─── Sample Examples (first 3) ───")
for i in range(3):
    sample = data["train"][i]
    print(f"\n[Sample {i+1}]")
    print(f"  Text  : {sample['text'][:120]}...")  # Truncate long text
    print(f"  Label : {sample['label']}")    # Raw label (integer)


─── Sample Examples (first 3) ───

[Sample 1]
  Text  : Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics,...
  Label : 2

[Sample 2]
  Text  : Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputat...
  Label : 2

[Sample 3]
  Text  : Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the ou...
  Label : 2


Understanding the label Mapping

In [ ]:
print("\n─── Label Mapping ───")
label_names = data["train"].features["label"].names
for idx, name in enumerate(label_names):
    print(f"  {idx} → {name}")


─── Label Mapping ───
  0 → World
  1 → Sports
  2 → Business
  3 → Sci/Tech


Class Distribution
Checking Data is balanced or not

In [ ]:
print("\n─── Class Distribution (train split) ───")
train_labels = data["train"]["label"]  # Get all labels as a flat list
counts = Counter(train_labels)


─── Class Distribution (train split) ───


In [ ]:
for label_id, count in sorted(counts.items()):
    class_name = label_names[label_id]
    bar = "█" * (count // 1000)   # Visual bar: 1 block per 1000 samples
    print(f"  {class_name:20s} | {count:6d} samples | {bar}")

  World                |  30000 samples | ██████████████████████████████
  Sports               |  30000 samples | ██████████████████████████████
  Business             |  30000 samples | ██████████████████████████████
  Sci/Tech             |  30000 samples | ██████████████████████████████


Analyzing Length of text

For BERT input length limit

In [ ]:
print("\n─── Text Length Statistics (train, first 5000 samples) ───")
lengths = [len(s["text"].split()) for s in data["train"].select(range(5000))]

avg_len = sum(lengths) / len(lengths)
max_len = max(lengths)
min_len = min(lengths)
over_128 = sum(1 for l in lengths if l > 128)  # Texts longer than common BERT max_length

print(f"  Average length : {avg_len:.1f} words")
print(f"  Min length     : {min_len} words")
print(f"  Max length     : {max_len} words")
print(f"  Over 128 words : {over_128} ({100*over_128/len(lengths):.1f}% of sample)")
print(f"\n  → Since most AG News headlines are short, we'll use max_length=128")
print(f"    when tokenizing. This saves memory and speeds up training.")


─── Text Length Statistics (train, first 5000 samples) ───
  Average length : 39.2 words
  Min length     : 11 words
  Max length     : 134 words
  Over 128 words : 3 (0.1% of sample)

  → Since most AG News headlines are short, we'll use max_length=128
    when tokenizing. This saves memory and speeds up training.


#PHASE 2: Tokenizing the Dataset

Imports

In [ ]:
from transformers import AutoTokenizer

Loading Data

In [ ]:
print("Loading dataset...")
data = load_dataset("ag_news")
label_names = data["train"].features["label"].names
print(f"Labels: {label_names}")

Loading dataset...
Labels: ['World', 'Sports', 'Business', 'Sci/Tech']


Loading the tokenizer

In [ ]:
MODEL_NAME = "bert-base-uncased"
print(f"\nLoading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Vocabulary size: {tokenizer.vocab_size:,} tokens")
print(f"Special tokens : [CLS]={tokenizer.cls_token_id}, "
      f"[SEP]={tokenizer.sep_token_id}, "
      f"[PAD]={tokenizer.pad_token_id}")


Loading tokenizer for bert-base-uncased...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocabulary size: 30,522 tokens
Special tokens : [CLS]=101, [SEP]=102, [PAD]=0


What tokenization looks like

By running one example manually before batch processing

In [ ]:
sample_text = "Wall St. Bears Claw Back Into the Black (Reuters)"
print(f"\n─── Tokenization Example ───")
print(f"Raw text : {sample_text}")

# tokenizer() returns a dict with input_ids, attention_mask, token_type_ids
encoding = tokenizer(sample_text)
print(f"input_ids: {encoding['input_ids']}")
# it starts with 101 ([CLS]), ends with 102 ([SEP])

# Convert IDs back to readable tokens — great for understanding subword splits
tokens = tokenizer.convert_ids_to_tokens(encoding['input_ids'])
print(f"tokens   : {tokens}")


─── Tokenization Example ───
Raw text : Wall St. Bears Claw Back Into the Black (Reuters)
input_ids: [101, 2813, 2358, 1012, 6468, 15020, 2067, 2046, 1996, 2304, 1006, 26665, 1007, 102]
tokens   : ['[CLS]', 'wall', 'st', '.', 'bears', 'claw', 'back', 'into', 'the', 'black', '(', 'reuters', ')', '[SEP]']


Preprocessing Function

In [ ]:
MAX_LENGTH = 128

def tokenize_function(examples):
    """
    Tokenize a batch of text samples.

    The 'examples' argument is a dict of lists (Hugging Face batched format):
        examples["text"]  = ["headline 1", "headline 2", ...]
        examples["label"] = [0, 2, ...]

    We return a new dict with the tokenizer outputs added.
    The label column stays untouched — it passes through automatically.
    """
    return tokenizer(
        examples["text"],
        padding="max_length",    # Pad shorter texts to max_length with [PAD]
        truncation=True,         # Cut longer texts at max_length
        max_length=MAX_LENGTH,   # Our chosen cap
    )

Applying Tokenization to Entire Dataset

In [ ]:
print(f"\nTokenizing dataset (max_length={MAX_LENGTH})...")
tokenized_data = data.map(tokenize_function, batched=True)
print("Done!")

# After .map(), each sample now has:
#   - text          (original, kept for reference)
#   - label         (0-3, untouched)
#   - input_ids     (list of 128 ints)
#   - attention_mask(list of 128 0s/1s)
#   - token_type_ids(list of 128 0s — all same sentence)
print("\nTokenized train sample keys:", list(tokenized_data["train"][0].keys()))


Tokenizing dataset (max_length=128)...


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Done!

Tokenized train sample keys: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


Setting Format for Pytorch

In [ ]:
tokenized_data.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "label"]
)

# Verify one sample
sample = tokenized_data["train"][0]
print("\n─── Tokenized Sample Shapes ───")
print(f"input_ids shape      : {sample['input_ids'].shape}")       # torch.Size([128])
print(f"attention_mask shape : {sample['attention_mask'].shape}")   # torch.Size([128])
print(f"label                : {sample['label']} "
      f"({label_names[sample['label'].item()]})")

# Quick sanity check: count real vs padding tokens in this sample
real_tokens = sample['attention_mask'].sum().item()
padding_tokens = MAX_LENGTH - real_tokens
print(f"\nReal tokens    : {real_tokens}")
print(f"Padding tokens : {padding_tokens}")
print("(Real = 1 in attention_mask, Padding = 0)")


─── Tokenized Sample Shapes ───
input_ids shape      : torch.Size([128])
attention_mask shape : torch.Size([128])
label                : 2 (Business)

Real tokens    : 41
Padding tokens : 87
(Real = 1 in attention_mask, Padding = 0)


Saving the Tokenized Data

In [ ]:
TOKENIZED_DATA_DIR = "/content/drive/MyDrive/tokenized_ag_news"

print(f"\nSaving tokenized dataset to {TOKENIZED_DATA_DIR} ...")
tokenized_data.save_to_disk(TOKENIZED_DATA_DIR)
print(f"Saved! Next time load with: load_from_disk('{TOKENIZED_DATA_DIR}')")


Saving tokenized dataset to /content/drive/MyDrive/tokenized_ag_news ...


Saving the dataset (0/1 shards):   0%|          | 0/120000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7600 [00:00<?, ? examples/s]

Saved! Next time load with: load_from_disk('/content/drive/MyDrive/tokenized_ag_news')


#PHASE 3: Fine-Tuning BERT

Imports

In [ ]:
import os
import torch
import numpy as np
from datasets import load_from_disk
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, f1_score

Mounting Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Configuration

In [ ]:
MODEL_NAME    = "bert-base-uncased"
NUM_LABELS    = 4
BATCH_SIZE    = 32
NUM_EPOCHS    = 3
LEARNING_RATE = 2e-5
LABEL_NAMES   = ["World", "Sports", "Business", "Sci/Tech"]
WARMUP_STEPS  = 1125

OUTPUT_DIR         = "/content/drive/MyDrive/bert_news_classifier"
TOKENIZED_DATA_DIR = "/content/drive/MyDrive/tokenized_ag_news"

Checking for existing check point

In [ ]:
def get_last_checkpoint(directory):
    if not os.path.exists(directory):
        return None
    checkpoints = [
        d for d in os.listdir(directory)
        if d.startswith("checkpoint-") and os.path.isdir(os.path.join(directory, d))
    ]
    if not checkpoints:
        return None
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    return os.path.join(directory, latest)

last_checkpoint = get_last_checkpoint(OUTPUT_DIR)
if last_checkpoint:
    print(f"✓ Resuming from: {last_checkpoint}")
else:
    print("Starting fresh training run.")

Starting fresh training run.


Loading the Preprocessed Data (The Tokenized Data)

In [ ]:
print(f"\nLoading tokenized dataset...")
tokenized_data = load_from_disk(TOKENIZED_DATA_DIR)
tokenized_data.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "label"]
)
train_dataset = tokenized_data["train"]
eval_dataset  = tokenized_data["test"]
print(f"Train: {len(train_dataset):,} | Test: {len(eval_dataset):,}")


Loading tokenized dataset...
Train: 120,000 | Test: 7,600


Loading the Model

In [ ]:
print(f"\nLoading {MODEL_NAME}...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label={0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"},
    label2id={"World": 0, "Sports": 1, "Business": 2, "Sci/Tech": 3},
)

# Saving tokenizer separately to be more cleaner and explicit.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)



Loading bert-base-uncased...


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Defining the Evaluation Metric Function

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average="macro")
    return {"accuracy": acc, "f1_macro": f1}

Configuring the training

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=2,
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
)

Building the trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

Training

In [ ]:
print("\nStarting training...")
print(f"  Device   : {'GPU ✓' if torch.cuda.is_available() else 'CPU'}")
print(f"  Resuming : {'Yes → ' + last_checkpoint if last_checkpoint else 'No'}\n")

trainer.train(resume_from_checkpoint=last_checkpoint)



Starting training...
  Device   : GPU ✓
  Resuming : No



Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.205532,0.171793,0.941447,0.941347
2,0.124069,0.161519,0.950132,0.950164
3,0.077938,0.191795,0.949079,0.949118


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=11250, training_loss=0.17049032777150472, metrics={'train_runtime': 2213.4414, 'train_samples_per_second': 162.643, 'train_steps_per_second': 5.083, 'total_flos': 2.368042020864e+16, 'train_loss': 0.17049032777150472, 'epoch': 3.0})

Evaluate

In [ ]:
print("\nFinal evaluation...")
results = trainer.evaluate()
print(f"  Accuracy : {results['eval_accuracy']*100:.2f}%")
print(f"  F1 Macro : {results['eval_f1_macro']:.4f}")


Final evaluation...


  Accuracy : 95.01%
  F1 Macro : 0.9502


Save model AND tokenizer explicitly

In [ ]:
print(f"\nSaving model to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)

print(f"Saving tokenizer to {OUTPUT_DIR}...")
tokenizer.save_pretrained(OUTPUT_DIR)


Saving model to /content/drive/MyDrive/bert_news_classifier...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving tokenizer to /content/drive/MyDrive/bert_news_classifier...


('/content/drive/MyDrive/bert_news_classifier/tokenizer_config.json',
 '/content/drive/MyDrive/bert_news_classifier/tokenizer.json')

 Verifying all required files exist

In [ ]:
print("\n─── Verifying saved files ───")

required_files = [
    "config.json",           # Model architecture + id2label mapping
    "tokenizer.json",        # Tokenizer vocabulary and rules
    "tokenizer_config.json", # Tokenizer settings
    "vocab.txt",             # WordPiece vocabulary
]

# Model weights: newer versions save as safetensors, older as pytorch_model.bin
weight_files = ["model.safetensors", "pytorch_model.bin"]

all_good = True

for fname in required_files:
    fpath = os.path.join(OUTPUT_DIR, fname)
    exists = os.path.exists(fpath)
    status = "✓" if exists else "✗ MISSING"
    print(f"  {status}  {fname}")
    if not exists:
        all_good = False

# Check that at least one weight file exists
weight_found = any(
    os.path.exists(os.path.join(OUTPUT_DIR, wf)) for wf in weight_files
)
if weight_found:
    found_name = next(wf for wf in weight_files
                      if os.path.exists(os.path.join(OUTPUT_DIR, wf)))
    print(f"  ✓  {found_name}  (model weights)")
else:
    print(f"  ✗ MISSING  model weights (neither safetensors nor pytorch_model.bin found)")
    all_good = False

# Check id2label is correctly written into config.json
import json
config_path = os.path.join(OUTPUT_DIR, "config.json")
if os.path.exists(config_path):
    with open(config_path) as f:
        config = json.load(f)
    id2label = config.get("id2label", {})
    print(f"\n  id2label in config.json: {id2label}")
    if id2label.get("0") == "World":
        print("  ✓  Label mapping is correct")
    else:
        print("  ✗  Label mapping is WRONG or missing — this causes the Sci/Tech bias bug")
        all_good = False

if all_good:
    print("\n✓ All files verified. Safe to close Colab.")
else:
    print("\n✗ Some files are missing. Do NOT close Colab yet.")
    print("  Run trainer.save_model(OUTPUT_DIR) and tokenizer.save_pretrained(OUTPUT_DIR) again.")


─── Verifying saved files ───
  ✓  config.json
  ✓  tokenizer.json
  ✓  tokenizer_config.json
  ✗ MISSING  vocab.txt
  ✓  model.safetensors  (model weights)

  id2label in config.json: {'0': 'World', '1': 'Sports', '2': 'Business', '3': 'Sci/Tech'}
  ✓  Label mapping is correct

✗ Some files are missing. Do NOT close Colab yet.
  Run trainer.save_model(OUTPUT_DIR) and tokenizer.save_pretrained(OUTPUT_DIR) again.


#PHASE 4: Evaluating the Model

Imports

In [ ]:
import os
import torch
import numpy as np
from datasets import load_from_disk
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import (
    classification_report,    # Per-class precision, recall, F1
    confusion_matrix,         # Class-by-class error breakdown
    accuracy_score,
    f1_score,
)


Config

In [ ]:
MODEL_DIR   = "/content/drive/MyDrive/bert_news_classifier"
LABEL_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
BATCH_SIZE  = 64

Pre-Flight Check

In [ ]:
# Verify the model directory has everything before trying to load.
# This catches the missing-file error with a clear message instead of a crash.
import json

required = ["config.json", "tokenizer.json", "tokenizer_config.json"]
missing  = [f for f in required if not os.path.exists(os.path.join(MODEL_DIR, f))]
weight_exists = any(
    os.path.exists(os.path.join(MODEL_DIR, w))
    for w in ["model.safetensors", "pytorch_model.bin"]
)

if missing or not weight_exists:
    print("✗ ERROR: Model directory is incomplete.")
    print(f"  Directory : {MODEL_DIR}")
    print(f"  Missing   : {missing + (['model weights'] if not weight_exists else [])}")
    print()
    print("  FIX: In phase3_train.py, make sure BOTH of these ran after training:")
    print("    trainer.save_model(OUTPUT_DIR)")
    print("    tokenizer.save_pretrained(OUTPUT_DIR)")
    raise SystemExit(1)

# Also verify id2label is correct — wrong mapping = biased predictions
with open(os.path.join(MODEL_DIR, "config.json")) as f:
    cfg = json.load(f)
id2label = cfg.get("id2label", {})
if id2label.get("0") != "World":
    print(f"✗ WARNING: id2label looks wrong: {id2label}")
    print("  Expected: {'0': 'World', '1': 'Sports', '2': 'Business', '3': 'Sci/Tech'}")
    print("  This will cause incorrect label names in predictions.")
else:
    print(f"✓ Label mapping verified: {id2label}")

print(f"✓ All required files found in {MODEL_DIR}")

✓ Label mapping verified: {'0': 'World', '1': 'Sports', '2': 'Business', '3': 'Sci/Tech'}
✓ All required files found in /content/drive/MyDrive/bert_news_classifier


Loading model and tokenizer

In [ ]:
print(f"\nLoading model from {MODEL_DIR}...")
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()   # Switch to eval mode: disables dropout, batch norm uses running stats
print(f"Running on: {device}")


Loading model from /content/drive/MyDrive/bert_news_classifier...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Running on: cuda


Loading Test Data

In [ ]:
print("\nLoading tokenized test data...")
tokenized_data = load_from_disk("./tokenized_ag_news")
tokenized_data.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "label"]
)
test_dataset = tokenized_data["test"]


Loading tokenized test data...


Running inference on test set

In [ ]:
# Process in batches to fit in memory.
# torch.no_grad() disables gradient computation — saves memory during eval.
# Only need forward pass, not backward.

print(f"\nRunning inference on {len(test_dataset):,} test samples...")
all_preds  = []
all_labels = []

from torch.utils.data import DataLoader
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

with torch.no_grad():
    for i, batch in enumerate(test_loader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"]

        # Forward pass: model returns logits, shape [batch_size, 4]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits

        # Convert logits to predicted class: argmax along class dimension
        preds = logits.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

        if (i + 1) % 20 == 0:
            print(f"  Processed {(i+1)*BATCH_SIZE:,} / {len(test_dataset):,}")

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
print("Done.")


Running inference on 7,600 test samples...
  Processed 1,280 / 7,600
  Processed 2,560 / 7,600
  Processed 3,840 / 7,600
  Processed 5,120 / 7,600
  Processed 6,400 / 7,600
Done.


Metric 1: Accuracy

In [ ]:
accuracy = accuracy_score(all_labels, all_preds)
print(f"\n─── Accuracy ───")
print(f"  {accuracy:.4f} ({accuracy*100:.2f}%)")
print("  (Correct predictions / total predictions)")


─── Accuracy ───
  0.9501 (95.01%)
  (Correct predictions / total predictions)


Metric 2: Macro F1

In [ ]:
f1_macro = f1_score(all_labels, all_preds, average="macro")
print(f"\n─── F1 Score (macro) ───")
print(f"  {f1_macro:.4f}")
print("  (Average F1 across all 4 classes equally)")



─── F1 Score (macro) ───
  0.9502
  (Average F1 across all 4 classes equally)


Metric 3: Per-class classification report

In [ ]:
print(f"\n─── Per-Class Classification Report ───")
print(classification_report(
    all_labels,
    all_preds,
    target_names=LABEL_NAMES
))



─── Per-Class Classification Report ───
              precision    recall  f1-score   support

       World       0.97      0.96      0.96      1900
      Sports       0.99      0.99      0.99      1900
    Business       0.93      0.91      0.92      1900
    Sci/Tech       0.91      0.94      0.93      1900

    accuracy                           0.95      7600
   macro avg       0.95      0.95      0.95      7600
weighted avg       0.95      0.95      0.95      7600



Metric 4: Confusion matrix

In [ ]:
print(f"─── Confusion Matrix ───")
cm = confusion_matrix(all_labels, all_preds)

# Pretty-print the matrix
header = f"{'':12}" + "".join(f"{n:12}" for n in LABEL_NAMES)
print(f"\n  Predicted →")
print(f"  {header}")
print(f"  {'True ↓':12}" + "─"*(12*4))
for i, row in enumerate(cm):
    row_str = f"  {LABEL_NAMES[i]:12}" + "".join(f"{val:12,}" for val in row)
    print(row_str)

print("\n  Interpretation:")
print("  Diagonal = correct predictions.")
print("  High off-diagonal value = those two classes get confused.")
print("  (World+Business confusion is common — both cover global economics.)")

─── Confusion Matrix ───

  Predicted →
              World       Sports      Business    Sci/Tech    
  True ↓      ────────────────────────────────────────────────
  World              1,816           9          40          35
  Sports                12       1,877           5           6
  Business              30           7       1,734         129
  Sci/Tech              18           8          80       1,794

  Interpretation:
  Diagonal = correct predictions.
  High off-diagonal value = those two classes get confused.
  (World+Business confusion is common — both cover global economics.)


Manual testing: trying my own examples

In [ ]:
print(f"\n─── Manual Prediction Test ───")
test_examples = [
    "SpaceX successfully launches Starlink satellite batch into orbit",
    "Federal Reserve raises interest rates amid inflation concerns",
    "Manchester United defeats Liverpool 3-1 in Premier League clash",
    "UN Security Council meets over escalating tensions in the Middle East",
]

for text in test_examples:
    # Tokenize single input
    inputs = tokenizer(
        text,
        return_tensors="pt",       # Return PyTorch tensors
        truncation=True,
        max_length=128,
        padding="max_length",
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model(**inputs)   # ** unpacks the dict as keyword args
        probs  = output.logits.softmax(dim=1)[0]   # Convert logits → probabilities
        pred   = probs.argmax().item()

    print(f"\n  Text      : {text[:60]}...")
    print(f"  Predicted : {LABEL_NAMES[pred]} ({probs[pred]*100:.1f}% confidence)")
    for i, (name, p) in enumerate(zip(LABEL_NAMES, probs)):
        bar = "▓" * int(p * 20)
        print(f"    {name:12} {p*100:5.1f}% {bar}")


─── Manual Prediction Test ───

  Text      : SpaceX successfully launches Starlink satellite batch into o...
  Predicted : Sci/Tech (99.2% confidence)
    World          0.4% 
    Sports         0.0% 
    Business       0.4% 
    Sci/Tech      99.2% ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

  Text      : Federal Reserve raises interest rates amid inflation concern...
  Predicted : Business (99.0% confidence)
    World          0.8% 
    Sports         0.0% 
    Business      99.0% ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    Sci/Tech       0.1% 

  Text      : Manchester United defeats Liverpool 3-1 in Premier League cl...
  Predicted : Sports (79.5% confidence)
    World         20.3% ▓▓▓▓
    Sports        79.5% ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    Business       0.1% 
    Sci/Tech       0.0% 

  Text      : UN Security Council meets over escalating tensions in the Mi...
  Predicted : World (99.7% confidence)
    World         99.7% ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    Sports         0.0% 
    Business       0.2% 
    Sci/Tech       0.1% 


#Phase 5: Deployment

Imports

In [ ]:
import gradio as gr

Config

In [ ]:
MODEL_DIR   = "/content/drive/MyDrive/bert_news_classifier"
LABEL_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
MAX_LENGTH  = 128

Loading Model and Tokenizer

In [ ]:
print(f"Loading model from {MODEL_DIR}...")
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()    # Disable dropout for consistent predictions
print(f"Model ready on {device}.")

Loading model from /content/drive/MyDrive/bert_news_classifier...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model ready on cuda.


Prediction Function

In [ ]:
# This is the core function Gradio will call on every user input.
# Inputs:  headline (str) — the text the user typed
# Outputs: dict mapping label → confidence (for Gradio's Label component)
#          This format is what gr.Label() expects for bar charts.

def predict(headline: str) -> dict:
    """
    Classify a news headline into one of 4 topics.
    Returns a dict of {class_name: probability} for display.
    """

    # Edge case: empty input
    if not headline.strip():
        return {name: 0.0 for name in LABEL_NAMES}

    # Step 1: Tokenize
    # return_tensors="pt" → PyTorch tensors
    # We don't need token_type_ids for single-sentence input, but including
    # it is harmless and makes the call signature consistent with training.
    inputs = tokenizer(
        headline,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

    # Move inputs to the same device as the model (CPU or GPU)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Step 2: Forward pass
    # torch.no_grad() skips gradient tracking — faster and less memory.
    # output.logits has shape [1, 4] (batch=1, classes=4)
    with torch.no_grad():
        output = model(**inputs)   # ** unpacks dict as keyword args
        logits = output.logits     # Raw scores, not probabilities yet

    # Step 3: Convert logits → probabilities
    # softmax makes the 4 scores sum to 1, each in range [0, 1]
    # .squeeze(0) removes the batch dimension: [1, 4] → [4]
    probs = logits.softmax(dim=1).squeeze(0).cpu().numpy()

    # Step 4: Build output dict for Gradio
    # gr.Label() expects {class_name: probability}
    return {LABEL_NAMES[i]: float(probs[i]) for i in range(len(LABEL_NAMES))}

Some try examples

In [ ]:
EXAMPLES = [
    "SpaceX launches new satellite constellation to provide global internet",
    "Manchester United manager resigns after dismal season performance",
    "Apple reports record quarterly earnings driven by iPhone sales",
    "World leaders gather at UN summit to address climate change",
    "Scientists discover potential cure for Alzheimer's disease",
    "Stock markets plunge as inflation fears grip investors",
]

Gradio Ui

In [ ]:
# gr.Interface() creates a full web UI from:
#   fn      = the function to call on each submission
#   inputs  = input component (a text box)
#   outputs = output component (a label with confidence bars)

demo = gr.Interface(
    fn=predict,

    inputs=gr.Textbox(
        label="News Headline",
        placeholder="Enter a news headline here...",
        lines=2,
    ),

    outputs=gr.Label(
        label="Topic Classification",
        num_top_classes=4,       # Show all 4 classes with confidence bars
    ),

    title="News Topic Classifier",
    description=(
        "Fine-tuned BERT model that classifies news headlines into "
        "World, Sports, Business, or Science/Technology categories. "
        "Trained on AG News (120,000 headlines)."
    ),

    examples=EXAMPLES,           # Clickable example buttons

    # Gradio's 'flagging' lets users report bad predictions.
    # We disable it for simplicity here.
    flagging_mode="never",
)

Launching

In [ ]:
# launch() starts a local web server.
# share=True creates a public URL via Gradio's tunneling service

if __name__ == "__main__":
    print("\nLaunching Gradio app...")
    demo.launch(
        share=True,   # Set to False if you don't want a public URL
        # server_port=7860  # Default port, change if occupied
    )



Launching Gradio app...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2628126e3933e688ef.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
